# Initial EDA - Customer Info

This notebook starts the initial exploratory data analysis for the customer segmentation project.

This phase focuses only on `customer_info`, the customer-level dataset that will be the base for segmentation because every customer must be included in the final clustering output.

## Imports and Data Loading

Load the CSV files directly with `pd.read_csv` from `../data/raw/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
customer_info = pd.read_csv("../data/raw/customer_info.csv")
customer_info.head()

In [ ]:
customer_basket = pd.read_csv("../data/raw/customer_basket.csv")
customer_basket.head()

## Basic Structure

Check the size, columns, data types, and summary statistics before making any assumptions about the data.

In [ ]:
print(f"Rows: {customer_info.shape[0]:,}")
print(f"Columns: {customer_info.shape[1]:,}")

customer_info.columns.to_frame(index=False, name="column")

In [ ]:
customer_info.dtypes.to_frame(name="dtype")

In [ ]:
customer_info.info()

In [ ]:
customer_info.describe()

In [ ]:
customer_info.select_dtypes(include=["object", "string"]).describe()

## Key Integrity Checks

`customer_id` must be reliable because the final output needs one assigned cluster for every customer.

In [ ]:
customer_id_checks = pd.DataFrame(
    {
        "check": [
            "duplicated customer_id count",
            "unique customer_id count",
            "customer_id is unique",
        ],
        "value": [
            customer_info["customer_id"].duplicated().sum(),
            customer_info["customer_id"].nunique(dropna=False),
            customer_info["customer_id"].is_unique,
        ],
    }
)

customer_id_checks

## Missing Values

Review missingness by column before deciding how to handle any values later.

In [ ]:
missing_summary = pd.DataFrame(
    {
        "missing_count": customer_info.isna().sum(),
        "missing_percentage": customer_info.isna().mean() * 100,
    }
).sort_values("missing_percentage", ascending=False)

missing_summary

In [ ]:
missing_with_values = missing_summary[missing_summary["missing_count"] > 0].reset_index(names="column")

if missing_with_values.empty:
    print("No missing values found.")
else:
    plt.figure(figsize=(10, 7))
    sns.barplot(
        data=missing_with_values,
        x="missing_percentage",
        y="column",
        color="steelblue",
    )
    plt.title("Missing Values by Column")
    plt.xlabel("Missing percentage")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.show()

Missingness is concentrated in `loyalty_card_number`, while several customer behavior fields have smaller missing shares. These columns should be handled carefully later, but no imputation or preprocessing is done in this notebook.

## Numerical Distributions

Plot the main numerical fields that may matter for customer segmentation. This is only inspection, not feature engineering.

In [ ]:
lifetime_spend_columns = [column for column in customer_info.columns if column.startswith("lifetime_spend_")]

numerical_columns = [
    "kids_home",
    "teens_home",
    "number_complaints",
    "distinct_stores_visited",
    *lifetime_spend_columns,
    "lifetime_total_distinct_products",
    "year_first_transaction",
    "percentage_of_products_bought_promotion",
    "typical_hour",
]
numerical_columns = [column for column in dict.fromkeys(numerical_columns) if column in customer_info.columns]

customer_info[numerical_columns].hist(bins=30, figsize=(16, 20), color="steelblue", edgecolor="white")
plt.suptitle("Numerical Distributions", y=1.01)
plt.tight_layout()
plt.show()

Spend columns appear right-skewed, which is common for customer purchasing data. Range checks below are more important at this stage than reshaping distributions.

## Categorical Distributions

Inspect simple categorical fields and visible name prefixes.

In [ ]:
customer_info["customer_gender"].value_counts(dropna=False)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=customer_info, x="customer_gender", color="steelblue")
plt.title("Customer Gender Counts")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
customer_info["loyalty_card_number"].value_counts(dropna=False)

In [ ]:
loyalty_status = customer_info["loyalty_card_number"].map({1.0: "Present"}).fillna("Missing")
loyalty_status_counts = loyalty_status.value_counts()

plt.figure(figsize=(6, 4))
sns.barplot(x=loyalty_status_counts.index, y=loyalty_status_counts.values, color="steelblue")
plt.title("Loyalty Card Number Presence")
plt.xlabel("Loyalty card number")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
name_prefix_counts = (
    customer_info["customer_name"]
    .astype("string")
    .str.extract(r"^(Bsc\.|Msc\.|Phd\.)", expand=False)
    .fillna("No visible prefix")
    .value_counts()
)

name_prefix_counts

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x=name_prefix_counts.index, y=name_prefix_counts.values, color="steelblue")
plt.title("Visible Customer Name Prefixes")
plt.xlabel("Prefix")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

Gender is balanced. `loyalty_card_number` looks more like an indicator than a meaningful numeric value, and the name prefix inspection is only a possible clue for later feature engineering.

## Data Quality Checks

Check for suspicious ranges without changing the data.

In [ ]:
promotion = customer_info["percentage_of_products_bought_promotion"]
year_first_transaction = customer_info["year_first_transaction"]
typical_hour = customer_info["typical_hour"]

range_checks = pd.DataFrame(
    {
        "check": [
            "promotion percentage outside [0, 1]",
            "year_first_transaction after 2026",
            "typical_hour outside [0, 23]",
            "negative kids_home",
            "negative teens_home",
            "negative number_complaints",
            "negative distinct_stores_visited",
        ],
        "count": [
            ((promotion < 0) | (promotion > 1)).sum(),
            (year_first_transaction > 2026).sum(),
            ((typical_hour < 0) | (typical_hour > 23)).sum(),
            (customer_info["kids_home"] < 0).sum(),
            (customer_info["teens_home"] < 0).sum(),
            (customer_info["number_complaints"] < 0).sum(),
            (customer_info["distinct_stores_visited"] < 0).sum(),
        ],
    }
)

range_checks

In [ ]:
spend_and_count_columns = lifetime_spend_columns + ["lifetime_total_distinct_products"]
negative_value_counts = customer_info[spend_and_count_columns].lt(0).sum().sort_values(ascending=False)

negative_value_counts[negative_value_counts > 0]

## Data Quality Notes

- Missing values are present, especially in `loyalty_card_number`; smaller missing shares also appear in several behavior and spend fields.
- Possible invalid values should be reviewed before preprocessing, especially promotion percentages outside `[0, 1]`.
- `loyalty_card_number` may be better treated as a loyalty flag later because the visible non-missing value is `1.0`.
- `customer_name` should not be used directly for modelling.
- Degree prefixes such as `Bsc.`, `Msc.`, and `Phd.` may become a useful engineered feature later, but no feature is created here.
- `year_first_transaction` includes suspicious future-looking values after 2026 that should be validated.
- `percentage_of_products_bought_promotion` includes invalid negative values and should be reviewed before modelling.